# 🎓 Educational Tutorial: ResNet18

## 1. History & Original Paper
* **Paper**: *Deep Residual Learning for Image Recognition* (He et al., 2015)
* **Concept**: Skip/Residual connections that bypass weights, solving the vanishing gradient problem.

## 2. Why GoogLeNet was Insufficient
As models got deeper, accuracy saturated and then degraded rapidly. ResNet solved degradation by redefining layers as learning residual functions with reference to the layer inputs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
from pathlib import Path
import time

print("Imports complete.")

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

In [ ]:
class ResNet18FromScratch(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.in_planes = 64
        
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        
        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        
    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
        
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

model = ResNet18FromScratch()
print("ResNet18 Parameter Count:", sum(p.numel() for p in model.parameters()))

In [ ]:
class NotebookEuroSATDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = self.root_dir / row["image_path"]
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        
        if self.transform:
            image = self.transform(image)
            
        return {
            "image": image,
            "label": label
        }

In [ ]:
PROJECT_ROOT = Path("../../")
csv_path = PROJECT_ROOT / "data/processed/train.csv"
val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

if not csv_path.exists():
    PROJECT_ROOT = Path(".").resolve().parents[1]
    csv_path = PROJECT_ROOT / "data/processed/train.csv"
    val_csv_path = PROJECT_ROOT / "data/processed/validation.csv"
    img_dir = PROJECT_ROOT / "data/raw/EuroSAT_RGB"

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

if csv_path.exists() and img_dir.exists():
    train_dataset = NotebookEuroSATDataset(csv_path, img_dir, train_transform)
    val_dataset = NotebookEuroSATDataset(val_csv_path, img_dir, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
    print("EuroSAT dataset splits loaded successfully.")
else:
    print("Warning: EuroSAT files not found! Mocking data loading.")
    class DummyDataset(Dataset):
        def __len__(self): return 16
        def __getitem__(self, idx): return {"image": torch.randn(3, 224, 224), "label": torch.randint(0, 10, (1,)).item()}
    train_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=True)
    val_loader = DataLoader(DummyDataset(), batch_size=4, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

model = ResNet18FromScratch().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

model.train()
start_time = time.time()
epoch_loss = 0.0
correct = 0
total = 0

print("Starting training verification step...")
max_batches = 2
for idx, batch in enumerate(train_loader):
    if idx >= max_batches: break
    images = batch["image"].to(device)
    labels = batch["label"].to(device)
    
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    epoch_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)
    
print(f"Verification training step complete in {time.time() - start_time:.2f}s | Loss: {epoch_loss/total:.4f} | Acc: {correct/total*100:.2f}%")